# 03 — MdB-Query in Brandwatch-konformer Form

Liest [`data/accounts.csv`](../data/accounts.csv), filtert auf `category == "MdB"`
und generiert **eine einzige** Brandwatch-Saved-Search.

**Nur** die Plattformen `x` / `instagram` / `facebook` — keine Websites, kein
YouTube/TikTok. Die Query wird global auf `language:de` eingeschränkt.

**Struktur** (Kommentare `<<< ... >>>` dienen der Lesbarkeit und werden laut
[`brandwatch_query_syntax.md`](./brandwatch_query_syntax.md) #9 vom Parser ignoriert):

1. Plattform-Reihenfolge: **X (Twitter)** → **Instagram** → **Facebook**
2. Innerhalb jeder Plattform: **Parteien alphabetisch** (AfD, CDU, CSU, Grüne,
   Linke, SPD, Sonstige Parteien)
3. Innerhalb der Partei: Handles alphabetisch

**Wichtig:** Es wird **kein** `site:`-Filter gesetzt. Authors auf X, IG und FB
leben in getrennten Handle-Räumen, d. h. `author:"..."` allein ist präzise.
Ein `site:`-Wrapper würde die soziale Daten nicht erfassen (Brandwatch indiziert
X/FB/IG-Mentions über Plattform-Integrationen, nicht über die Web-Domain).

**Output:** `output/queries/MdB_query.txt` (genau eine Datei).

In [1]:
import os
import shutil

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
QUERIES_DIR = os.path.join(PROJECT_ROOT, "output", "queries")

ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")
OUTPUT_FILE = os.path.join(QUERIES_DIR, "MdB_query.txt")

# Plattform-Reihenfolge (channel_key, hübsches Label).
PLATFORM_ORDER = [
    ("x",         "X (Twitter)"),
    ("instagram", "Instagram"),
    ("facebook",  "Facebook"),
]

# Parteien-Reihenfolge: alphabetisch, Umlaute deutsch-üblich einsortiert.
PARTY_ORDER = ["AfD", "CDU", "CSU", "Grüne", "Linke", "SPD", "Sonstige Parteien"]

LANGUAGE_FILTER = "language:de"

# Altes Output-Verzeichnis platt machen, damit keine Chunk-Dateien aus früheren Läufen verbleiben.
if os.path.isdir(QUERIES_DIR):
    shutil.rmtree(QUERIES_DIR)
os.makedirs(QUERIES_DIR, exist_ok=True)

## 1. MdB-Einträge laden und auf erlaubte Plattformen filtern

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)
allowed_channels = {p[0] for p in PLATFORM_ORDER}

mdb = (
    accounts[(accounts["category"] == "MdB") & (accounts["channel"].isin(allowed_channels))]
    .dropna(subset=["handle", "label"])
    .drop_duplicates(subset=["channel", "handle"])
    .copy()
)

print(f"MdB-Einträge (x / instagram / facebook, dedupliziert): {len(mdb)}")
print(mdb.groupby(["channel", "label"]).size().unstack(fill_value=0))

unknown_parties = sorted(set(mdb["label"].unique()) - set(PARTY_ORDER))
if unknown_parties:
    print(f"\n⚠  Unbekannte Parteien (werden ans Ende sortiert): {unknown_parties}")

MdB-Einträge (x / instagram / facebook, dedupliziert): 1440
label      AfD  CDU  CSU  Grüne  Linke  SPD  Sonstige Parteien
channel                                                       
facebook   122  153   39     75     32  119                  2
instagram   95  157   42     85     58  118                  1
x           81   75   23     65     28   69                  1


## 2. Helper: Brandwatch-Formatierung

In [3]:
def bw_author(handle: str) -> str:
    """Ein Eintrag als `author:"..."` — immer gequotet wegen Punkten/Bindestrichen in Slugs."""
    h = str(handle).strip().replace('"', '\\"')
    return f'author:"{h}"'


def indent(text: str, spaces: int) -> str:
    pad = " " * spaces
    return "\n".join(pad + line if line else line for line in text.split("\n"))


def party_block(party: str, platform_label: str, handles) -> str:
    """Eine Partei-Gruppe innerhalb einer Plattform: `<<< ... >>>` + geklammerter OR-Block."""
    body = "\n  OR ".join(bw_author(h) for h in handles)
    return (
        f"<<< {platform_label} — {party} — {len(handles)} Handles >>>\n"
        f"(\n  {body}\n)"
    )


def platform_block(platform_key: str, platform_label: str, df_platform: pd.DataFrame) -> str:
    """Ein Plattform-Block: geklammerter OR-Block aus Partei-Sub-Blöcken."""
    # Parteien-Reihenfolge: bekannte zuerst in PARTY_ORDER-Reihenfolge, unbekannte alphabetisch dahinter.
    present = set(df_platform["label"].unique())
    ordered_parties = [p for p in PARTY_ORDER if p in present]
    ordered_parties += sorted(present - set(PARTY_ORDER))

    party_blocks = []
    for party in ordered_parties:
        handles = (
            df_platform[df_platform["label"] == party]["handle"]
            .sort_values(key=lambda s: s.str.lower())
            .tolist()
        )
        party_blocks.append(party_block(party, platform_label, handles))

    inner = "\n\nOR\n\n".join(party_blocks)
    total = len(df_platform)
    return (
        f"<<< {platform_label} — {total} Handles — {len(ordered_parties)} Parteien >>>\n"
        f"(\n"
        f"{indent(inner, 2)}\n"
        f")"
    )

## 3. Plattform-Blöcke bauen

In [4]:
platform_sections = []

for platform_key, platform_label in PLATFORM_ORDER:
    df_p = mdb[mdb["channel"] == platform_key]
    if df_p.empty:
        print(f"{platform_label}: keine Handles — übersprungen")
        continue
    block = platform_block(platform_key, platform_label, df_p)
    platform_sections.append(block)
    print(f"{platform_label}: {len(df_p)} Handles, {df_p['label'].nunique()} Parteien")

print(f"\nPlattform-Blöcke: {len(platform_sections)}")

X (Twitter): 342 Handles, 7 Parteien
Instagram: 556 Handles, 7 Parteien
Facebook: 542 Handles, 7 Parteien

Plattform-Blöcke: 3


## 4. Gesamt-Query zusammensetzen

Schema:
```
<<< MdB — Gesamt-Query — N Handles >>>
(
  language:de
  AND
  (
    <<< X (Twitter) — … >>>
    ( <Partei-OR-Blöcke> )
    OR
    <<< Instagram — … >>>
    ( <…> )
    OR
    <<< Facebook — … >>>
    ( <…> )
  )
)
```

In [5]:
total_handles = len(mdb)
header_comment = f"<<< MdB — Gesamt-Query — {total_handles} Handles — {len(platform_sections)} Plattformen >>>"

platforms_joined = "\n\nOR\n\n".join(platform_sections)

query = (
    f"{header_comment}\n"
    f"(\n"
    f"  {LANGUAGE_FILTER}\n"
    f"  AND\n"
    f"  (\n"
    f"{indent(platforms_joined, 4)}\n"
    f"  )\n"
    f")\n"
)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(query)

print(f"Datei: {OUTPUT_FILE}")
print(f"Größe: {os.path.getsize(OUTPUT_FILE):,} bytes")
print(f"Handles gesamt: {total_handles}")
print(f"Plattform-Blöcke: {len(platform_sections)}")

Datei: /Users/zorbeyozcan/Projekte/query_printer/output/queries/MdB_query.txt
Größe: 52,217 bytes
Handles gesamt: 1440
Plattform-Blöcke: 3


## 5. Preview — erste und letzte Zeilen

In [6]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Gesamt: {len(lines)} Zeilen\n")
print("--- erste 40 Zeilen ---")
print("".join(lines[:40]))
print("--- letzte 20 Zeilen ---")
print("".join(lines[-20:]))

Gesamt: 1579 Zeilen

--- erste 40 Zeilen ---
<<< MdB — Gesamt-Query — 1440 Handles — 3 Plattformen >>>
(
  language:de
  AND
  (
    <<< X (Twitter) — 342 Handles — 7 Parteien >>>
    (
      <<< X (Twitter) — AfD — 81 Handles >>>
      (
        author:"adambalten"
        OR author:"afdprotschka"
        OR author:"AlexanderWolfHH"
        OR author:"alice_weidel"
        OR author:"andreasbleckmdb"
        OR author:"AndreasMayerAfD"
        OR author:"AngelaRudzka"
        OR author:"arne22667953"
        OR author:"B_Treuheit"
        OR author:"Beatrix_vStorch"
        OR author:"BirgitBessin"
        OR author:"Carina_Schiessl"
        OR author:"chrwirthmdb"
        OR author:"dario_seifert"
        OR author:"denispauliafd"
        OR author:"DirkBrandes74"
        OR author:"Dr_Rainer_Kraft"
        OR author:"DrBerndBaumann"
        OR author:"drbirghan"
        OR author:"DrChristinaBaum"
        OR author:"DrMEspendiller"
        OR author:"DrRothfuss"
        OR author:"e